In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
csv_path = Path("../data/processed data/sales_cleaned_data.csv")

print("File exists:", csv_path.exists())
print("Full path:", csv_path.resolve())

File exists: True
Full path: C:\Users\yuvra\OneDrive\Desktop\DA projects\Amazon sales analysis\data\processed data\sales_cleaned_data.csv


In [3]:
#reading csv file
df = pd.read_csv(csv_path)

In [4]:
# Database connection
conn = sqlite3.connect("../sql/amazon_analytics.db")

# importing csv data to db
df.to_sql(
    name="sales",
    con=conn,
    if_exists="replace",
    index=False
)

print("CSV data successfully imported")

CSV data successfully imported


In [5]:
#checking connection is working or not
query = """
SELECT COUNT(*) AS total_rows
FROM sales;
"""

display(pd.read_sql_query(query, conn))

,total_rows
0,128943


In [6]:
query = """
SELECT *
FROM sales
LIMIT 1;
"""

display(pd.read_sql_query(query, conn))

,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,b2b
0,171-7326790-9044369,2022-03-31,Shipped,Amazon,Amazon.in,Expedited,JNE3614,JNE3614-KR-L,Kurta,L,B098133PV7,Shipped,1,INR,453.0,ARANI TIRUVANNAMALAI DISTRICT,TAMIL NADU,632317,IN,0


In [7]:
#Q-1 How many total records, unique orders, and total units ordered are present in the dataset?
query = """
    select 
    count(*) as total_record,
    count(DISTINCT order_id) as unique_record,
    sum(qty) as total_orders
    from sales;
    
"""
result = pd.read_sql_query(query, conn)
display(result)

,total_record,unique_record,total_orders
0,128943,120352,116648


In [8]:
#Q3. Which product categories generate the highest sales and order volumes?
query = """
    select 
    category,
    sum(amount) as total_sales,
    count(order_id) as total_orders
    from sales
    GROUP BY category
    ORDER BY total_sales desc;
    
"""
result = pd.read_sql_query(query, conn)
display(result)

,category,total_sales,total_orders
0,Set,41295773.17,50272
1,Kurta,23341914.20,49864
2,Western Dress,11728629.69,15495
3,Top,5643606.80,10621
4,Ethnic Dress,833163.66,1158
5,Blouse,487590.68,926
6,Bottom,163637.98,440
7,Saree,129770.26,164
8,Dupatta,915.00,3


In [14]:
#Q4- Which product categories generate the highest sales revenue?
query = """
        select 
            category, 
            sum(amount) as total_sales,
            sum(qty) as total_quantity
        FROM sales
        GROUP BY category
        ORDER BY total_sales DESC;
"""
result = pd.read_sql_query(query, conn)
display(result)

,category,total_sales,total_quantity
0,Set,41295773.17,45286
1,Kurta,23341914.20,45048
2,Western Dress,11728629.69,13943
3,Top,5643606.80,9902
4,Ethnic Dress,833163.66,1052
5,Blouse,487590.68,864
6,Bottom,163637.98,398
7,Saree,129770.26,152
8,Dupatta,915.00,3


In [15]:
#Q-5 How does the company's sales performance change over time, and which months generate the highest sales?
query = """
        select 
            strftime('%Y-%m', date) AS sales_month,
            sum(amount) as total_sales,
            ship_state
            
        FROM sales
        GROUP BY sales_month
        ORDER BY sales_month DESC;

"""
result = pd.read_sql_query(query, conn)
display(result)


,sales_month,total_sales,ship_state
0,2022-06,24896697.88,TELANGANA
1,2022-05,27843931.25,TAMIL NADU
2,2022-04,30776851.96,MAHARASHTRA
3,2022-03,107520.35,TAMIL NADU


In [24]:
#Q-6 What percentage of total orders are cancelled, and which product categories experience the highest cancellation rates?
query = """
        select 
            category,
            count(order_id) as total_orders,
            SUM(CASE
                    WHEN status = "Cancelled" THEN 1
                    ELSE 0
                END
            ) as total_cancelled_orders,
            ROUND(
                SUM(CASE
                        WHEN status = 'Cancelled' THEN 1
                        ELSE 0
                    END) * 100.0 / COUNT(order_id)
                ) AS cancelled_orders_percentage
        FROM sales
        GROUP BY category 
        ORDER BY cancelled_orders_percentage DESC;
"""

result = pd.read_sql_query(query, conn)
display(result)

,category,total_orders,total_cancelled_orders,cancelled_orders_percentage
0,Set,50272,7337,15.0
1,Kurta,49864,7257,15.0
2,Western Dress,15495,2125,14.0
3,Bottom,440,60,14.0
4,Saree,164,21,13.0
5,Ethnic Dress,1158,146,13.0
6,Blouse,926,116,13.0
7,Top,10621,1276,12.0
8,Dupatta,3,0,0.0


In [29]:
#Q-7 Which Indian states generate the highest sales revenue and contribute the most to the company's overall order volume?

query = """
        select
            ship_state,
            sum(amount) as total_sales,
            count(order_id) as total_orders,
            ROUND(SUM(amount) * 1.0 /
                COUNT(order_id),2
                )AS average_order_value
            FROM sales
            GROUP BY ship_state
            ORDER BY total_orders DESC;
            
"""
result = pd.read_sql_query(query, conn)
display(result)


,ship_state,total_sales,total_orders,average_order_value
0,MAHARASHTRA,14102086.64,22256,633.63
1,KARNATAKA,11081688.87,17320,639.82
2,TAMIL NADU,6951015.11,11481,605.44
3,TELANGANA,7364087.65,11327,650.14
4,UTTAR PRADESH,7261992.58,10635,682.84
5,DELHI,4603143.46,6964,660.99
6,KERALA,4111028.08,6584,624.40
7,WEST BENGAL,3776152.94,5961,633.48
8,ANDHRA PRADESH,3462475.22,5429,637.77
9,GUJARAT,2946547.82,4489,656.39


In [35]:
#Q-8 Which product sizes have the highest demand, and how does product demand vary across different product categories?

query = """
        select
            category,
            size,
            sum(qty) as total_quantity_sold,
            count(order_id) as total_orders
            FROM sales
            GROUP BY size
            ORDER BY total_orders DESC;
            
"""
result = pd.read_sql_query(query, conn)
display(result)

,category,size,total_quantity_sold,total_orders
0,Set,M,20449,22703
1,Kurta,L,19993,22122
2,Set,XL,18928,20878
3,Top,XXL,16511,18089
4,Set,S,15337,17092
5,Kurta,3XL,13525,14810
6,Set,XS,9939,11157
7,Kurta,6XL,688,737
8,Set,5XL,513,550
9,Kurta,4XL,397,426


In [36]:
#Q9- Which are the top 3 highest-revenue-generating products within each product category?

query = """SELECT
    category,
    sku,
    total_sales,
    total_quantity,
    product_rank

FROM (
    SELECT
        category,
        sku,

        SUM(amount) AS total_sales,

        SUM(qty) AS total_quantity,

        DENSE_RANK() OVER (
            PARTITION BY category
            ORDER BY SUM(amount) DESC
        ) AS product_rank

    FROM sales

    WHERE status = 'Shipped - Delivered to Buyer'

    GROUP BY category, sku
)

WHERE product_rank <= 3

ORDER BY category, product_rank;

"""
result = pd.read_sql_query(query, conn)
display(result)


,category,sku,total_sales,total_quantity,product_rank
0,Blouse,J0217-BL-S,7315.0,11,1
1,Blouse,J0217-BL-L,6864.0,10,2
2,Blouse,J0217-BL-XL,3990.0,7,3
3,Bottom,AN211-BLACK-L,1806.0,6,1
4,Bottom,BTM039-PP-M,1800.0,5,2
5,Bottom,BTM039-PP-L,1800.0,5,2
6,Bottom,BTM038-PP-XXXL,1512.0,4,3
7,Ethnic Dress,J0006-SET-L,13309.0,15,1
8,Ethnic Dress,J0006-SET-S,9349.0,11,2
9,Ethnic Dress,J0209-DR-L,8471.0,12,3
